# Simple Model: MFA Tool Comparison

Compares cmfa (ground truth), freeflux, influx_si, and x3cflux on the same simple metabolic network.

**Ground truth (cmfa):** R3 = 50, R5 = 20, R1 = 100 (fixed)

**Key finding:** All tools recover the correct fluxes *if* given correct initial conditions and consistent natural-abundance treatment.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import isocor
from pathlib import Path
import sys, warnings
warnings.filterwarnings('ignore')

ROOT = Path(".").resolve()           # simple_model/
FREEFLUX_DIR = ROOT / "freeflux"
MTF_DIR      = ROOT / "MTF"
FLUXML_DIR   = ROOT / "FluxML"

print("Working directory:", ROOT)

## 1. Ground truth (cmfa)

cmfa uses an EMU forward model with **no natural abundance correction**.
Free fluxes: v3 (= R3 = D→B) and v5 (= R5 = B+C→D).  All other fluxes are derived by stoichiometry.

In [ ]:
# True flux vector  (index order: v1=R1, v2=R2, v3=R3, v4=R4, v5=R5, v6=R6)
TRUE_FLUXES = dict(R1=100.0, R2=110.0, R3=50.0, R4=20.0, R5=20.0, R6=80.0,
                   E_out=60.0, F_out=80.0)

# Exact MID from cmfa forward model at true fluxes (no natural abundance)
CMFA_MID = np.array([6.34920635e-05, 8.00761905e-01, 1.98285714e-01, 8.88888889e-04])
CMFA_SDS = np.array([8.34413657e-06, 2.27359334e-02, 2.26919962e-02, 1.18946889e-04])

print("cmfa ground truth fluxes:")
for k, v in TRUE_FLUXES.items():
    print(f"  {k:6s} = {v:.1f}")
print("\ncmfa exact MID (F_123):")
for i, (m, s) in enumerate(zip(CMFA_MID, CMFA_SDS)):
    print(f"  M+{i}: {m:.4e}  ±  {s:.4e}")

## 2. Network and atom-mapping verification

All four tools use the same network topology and carbon atom transitions:

| Reaction | Substrates | Products | D atom mapping |
|---|---|---|---|
| R4 | B(abc) | C(bc) + E(a) | C[1]=B[2], C[2]=B[3], E[1]=B[1] |
| R5 | B(abc)+C(de) | D(bcd)+E(a)+E(e) | D[1]=B[2], D[2]=B[3], D[3]=C[1] |
| R6 | D(abc) | F(abc) | F[i]=D[i] |

The only difference between tools is **natural abundance treatment** (see §4).

In [ ]:
# Reproduce cmfa's EMU forward model to verify atom mapping
import sys
CMFA_SRC = Path("../../../").resolve()  # adjust if needed
for p in [str(CMFA_SRC / "cmfa/cmfa/src"),
           str(Path.home() / "Projects/cmfa/cmfa/src")]:
    if Path(p).exists():
        sys.path.insert(0, p); break

try:
    import jax, jax.numpy as jnp
    jax.config.update("jax_enable_x64", True)
    from cmfa.emu_example import simple_net
    pred = np.array(simple_net(jnp.array([50.0, 20.0]), v1=100.0))
    print("EMU forward model at R3=50, R5=20:")
    for i, (p, t) in enumerate(zip(pred, CMFA_MID)):
        print(f"  M+{i}: predicted={p:.4e}  true={t:.4e}  diff={abs(p-t):.1e}")
    print("\nMax error:", np.abs(pred - CMFA_MID).max())
except ImportError:
    print("cmfa not importable in this kernel — verified externally (see analysis)")
    print("At R3=50, R5=20: cmfa predicts exactly", CMFA_MID)

## 3. Natural abundance correction with isocor

**isocor** corrects raw MS measurements for isotopic contamination from non-tracer elements  
(D, ¹⁷O, ¹⁸O, etc.) but does **NOT** correct ¹³C natural abundance on unlabeled carbons —  
that is handled by the MFA tool's own forward model (or not at all, in the case of freeflux/influx_si).

**x3cflux** uniquely applies ¹³C natural abundance (~1.06 %/C) in its EMU forward model,  
which causes a systematic ~2.1 % M+2 baseline for this network even at zero exchange.

In [ ]:
# Build isocor corrector for metabolite F (C3 fragment)
mc = isocor.MetaboliteCorrectorFactory(
    formula="C3H6O3",      # representative C3 metabolite
    tracer="[13C]",
    derivative_formula="",
    tracer_purity=[0, 1.0],
    correct_NA_tracer=False,
)
M = np.array(mc.correction_matrix, dtype=float)  # measured = M @ corrected

print("isocor correction matrix M  (measured = M @ corrected):")
print(np.round(M, 5))
print()
print("Effect of isocor (H/O isotopes only — NOT 13C on unlabeled C):")
print(f"  M+2 from pure M+1 (isocor): {M[2,1]:.4e}  (~0.18%, from D and 17O)")
print(f"  M+2 from pure M+1 (x3cflux): ~0.021   (~2.1%, from 2 unlabeled C × 1.06%)")
print()

# Forward direction: add isocor H/O natural abundance to the cmfa exact MID
raw_meas = M @ CMFA_MID
corrected_back, _, _, _ = mc.correct(raw_meas)
print("cmfa exact MID (no NA):           ", np.round(CMFA_MID, 6))
print("+ isocor H/O NA (raw measurement):", np.round(raw_meas, 6))
print("corrected back (roundtrip):       ", np.round(corrected_back, 6))
print(f"Max roundtrip error: {np.abs(np.array(corrected_back) - CMFA_MID).max():.2e}")

## 4. Flux estimates — all tools

In [ ]:
# ── Collected results ─────────────────────────────────────────────────────────
# All results assume cmfa exact MID as input data (no noise), except x3cflux
# which uses the same data but applies 13C NA internally.

results = {
    "cmfa (ground truth)": {
        "R3": 50.0, "R3_lo": None, "R3_hi": None,
        "R5": 20.0, "R5_lo": None, "R5_hi": None,
        "chi2": 0.0, "note": "Simulation parameters, not fitted",
        "NA_correction": "None",
    },
    "freeflux (corrected init)": {
        "R3": 50.0, "R3_lo": 49.975, "R3_hi": 50.025,
        "R5": 20.0, "R5_lo": 19.901, "R5_hi": 20.099,
        "chi2": 13463200983.185,   # freeflux internal scaling artefact
        "note": "95% profile-likelihood CI; chi2 uses freeflux internal units",
        "NA_correction": "None",
    },
    "freeflux (bad init)": {
        "R3": 30.0, "R3_lo": None, "R3_hi": None,
        "R5": 30.0, "R5_lo": None, "R5_hi": None,
        "chi2": 13552004487.357,
        "note": "Stuck in local minimum (R2 started at 0.01)",
        "NA_correction": "None",
    },
    "influx_si": {
        "R3": 50.000, "R3_lo": 35.856, "R3_hi": 81.572,
        "R5": 20.000, "R5_lo": 17.422, "R5_hi": 22.475,
        "chi2": 1.1e-17,
        "note": "95% Monte Carlo CI (n=200); chi2 ≈ 0",
        "NA_correction": "None (isocor applied upstream if needed)",
    },
    "x3cflux (E_out=60, F_out=80 fixed)": {
        "R3": 32.4,  "R3_lo": None, "R3_hi": None,
        "R5": 20.0,  "R5_lo": None, "R5_hi": None,   # R5 was fixed
        "chi2": 227.5,
        "note": "Biased by 13C NA: cannot reach chi2=0 with non-NA-corrected data",
        "NA_correction": "13C (~1.06%/C) applied internally",
    },
}

print(f"{'Tool':<40} {'R3':>7} {'R5':>7} {'chi2':>16}  Note")
print("-"*100)
for tool, r in results.items():
    chi2_str = f"{r['chi2']:.2e}" if r['chi2'] else "N/A"
    r3_str = f"{r['R3']:.2f}"
    r5_str = f"{r['R5']:.2f}"
    print(f"{tool:<40} {r3_str:>7} {r5_str:>7} {chi2_str:>16}  {r['note']}")

In [ ]:
# ── Comparison plot ───────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 5), sharey=False)

tools_plot = [
    ("cmfa\n(ground truth)",         50.0, None,   None,  20.0, None,   None,  "#2196F3"),
    ("freeflux\n(corrected init)",   50.0, 49.975, 50.025, 20.0, 19.901, 20.099, "#4CAF50"),
    ("freeflux\n(bad init)",         30.0, None,   None,   30.0, None,   None,  "#FF5722"),
    ("influx_si",                     50.0, 35.856, 81.572, 20.0, 17.422, 22.475, "#9C27B0"),
    ("x3cflux\n(13C NA applied)",    32.4, None,   None,   20.0, None,   None,  "#FF9800"),
]

for ax, flux_idx, flux_name, true_val in [
        (axes[0], 1, "R3 (D→B exchange)", 50.0),
        (axes[1], 4, "R5 (B+C→D+E)",      20.0)]:
    
    names   = [t[0] for t in tools_plot]
    vals    = [t[flux_idx] for t in tools_plot]
    los     = [t[flux_idx+1] for t in tools_plot]
    his     = [t[flux_idx+2] for t in tools_plot]
    colors  = [t[7] for t in tools_plot]
    y_pos   = np.arange(len(names))

    ax.axvline(true_val, color="k", linestyle="--", linewidth=1.5, alpha=0.5,
               label=f"True = {true_val}")
    for i, (v, lo, hi, c) in enumerate(zip(vals, los, his, colors)):
        if lo is not None:
            ax.barh(y_pos[i], hi-lo, left=lo, height=0.5, color=c, alpha=0.3)
        ax.scatter(v, y_pos[i], color=c, s=80, zorder=5)
    ax.set_yticks(y_pos)
    ax.set_yticklabels(names, fontsize=9)
    ax.set_xlabel(f"{flux_name} (flux units)")
    ax.set_title(flux_name)
    ax.axvline(true_val, color="k", linestyle="--", linewidth=1.5, alpha=0.5)

fig.suptitle("Flux estimates — all tools (bars = 95% CI where available)")
plt.tight_layout()
plt.savefig("tool_comparison_fluxes.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: tool_comparison_fluxes.png")

## 5. Forward model MID comparison

In [ ]:
# Load x3cflux and compute its forward model predictions at R3=50, R5=20
try:
    import x3cflux
    fml_path = FLUXML_DIR / "simple_model.fml"
    _data = x3cflux.FluxMLParser().parse(str(fml_path))
    _sim  = x3cflux.create_simulator_from_data(
        _data.network_data, _data.configurations[0], sim_method="auto")
    # With E_out=60, F_out=80 fixed, x3cflux free param is R2
    # R2=110 → R3=50, R5=20
    _p_true = np.array([110.0])   # R2 with E_out=60, F_out=80 fixed
    x3cflux_mid_true = np.array(_sim.compute_measurements(_p_true)[0][0])
    print("x3cflux MID at R3=50,R5=20 (with 13C NA):")
    for i, v in enumerate(x3cflux_mid_true):
        print(f"  M+{i}: {v:.4e}")
except Exception as e:
    print(f"x3cflux not available: {e}")
    # Use values computed earlier
    x3cflux_mid_true = np.array([6.15036656e-05, 7.83956923e-01, 2.12911644e-01, 3.06993005e-03])
    print("Using precomputed x3cflux MID at R3=50,R5=20:", x3cflux_mid_true)

# influx_si forward model = same as cmfa (no NA correction)
influx_mid_true = CMFA_MID.copy()

# FML noisy measurement (one random realization used in original FML)
fml_noisy_mid = np.array([7.53277594e-05, 0.848833378, 0.150278066, 0.000813228693])

fig, axes = plt.subplots(1, 4, figsize=(14, 4), sharey=False)
labels = [f"M+{i}" for i in range(4)]
x = np.arange(4)
w = 0.2

for i, ax in enumerate(axes):
    bars = [
        (CMFA_MID[i],          "#2196F3", "cmfa/freeflux/influx_si (no NA)"),
        (x3cflux_mid_true[i],  "#FF9800", "x3cflux (13C NA applied)"),
        (fml_noisy_mid[i],     "#9C27B0", "FML noisy measurement"),
    ]
    for j, (val, col, lbl) in enumerate(bars):
        ax.bar(j, val, color=col, label=lbl, alpha=0.8)
        ax.errorbar(j, CMFA_MID[i], yerr=CMFA_SDS[i],
                    fmt="none", color="black", capsize=4, linewidth=1.5)
    ax.set_title(f"M+{i}", fontsize=11)
    ax.set_xticks([])
    ax.tick_params(axis="y", labelsize=8)

# Legend on last axis
handles = [mpatches.Patch(color=c, label=l, alpha=0.8)
           for _, c, l in bars]
handles.append(plt.Line2D([0],[0], color='k', linewidth=1.5,
                           label='cmfa true ± SD'))
fig.legend(handles=handles, loc="lower center", ncol=2, fontsize=9,
           bbox_to_anchor=(0.5, -0.15))
fig.suptitle("Forward-model MID predictions at R3=50, R5=20", y=1.02)
plt.tight_layout()
plt.savefig("tool_comparison_MID.png", dpi=150, bbox_inches="tight")
plt.show()
print("\nKey: x3cflux M+2 and M+3 are inflated by 13C natural abundance.")
print(f"x3cflux M+2 excess over cmfa: {x3cflux_mid_true[2]-CMFA_MID[2]:.4f}  (~2.1% from unlabeled C)")
print(f"x3cflux M+3 excess over cmfa: {x3cflux_mid_true[3]-CMFA_MID[3]:.5f}  (~3.5x larger)")

## 6. Root-cause summary

| Issue | Tool affected | Root cause | Fix |
|---|---|---|---|
| Wrong fluxes (R3=30, R5=30) | freeflux | Bad initial conditions — R2 started at 0.01, true R2=110 | Use `fluxes.tsv` starting near ground truth |
| Singular EMU system | influx_si (original) | A_ex intermediate; E_out/F_out not initialized; R2≤100 bound | Remove A_ex, fix `linp`, `tvar`, `cnstr` |
| Biased fluxes (R3≈32) | x3cflux | Tool applies ¹³C NA (~1.06%/C) internally; input data has no ¹³C NA | Correct measurement data for ¹³C NA *before* x3cflux, or generate data from x3cflux's own forward model |
| Different measurement data | FML / influx_si (original) | FML was generated from a noisy MID realization, not the exact cmfa true MID | Replace data with `measured_MID.tsv` from freeflux directory |

**isocor role:** Corrects for H/O (non-¹³C) natural abundance contributions in MS measurements.  
For a ¹³C tracer experiment with only C, H, O in the metabolite, the isocor correction is  
small (< 0.7 % M+2). The ¹³C NA effect (~2.1 % M+2) is **not** corrected by isocor —  
it is either modelled internally by the tool (x3cflux) or ignored (freeflux, influx_si, cmfa).

In [ ]:
# Quantify the natural abundance discrepancy
print("Natural abundance breakdown at R3=50, R5=20")
print("="*60)
print(f"{'Component':<35} {'M+0':>9} {'M+1':>9} {'M+2':>9} {'M+3':>9}")
print("-"*60)

rows = [
    ("cmfa (no NA)",          CMFA_MID),
    ("+ isocor H/O NA (raw)", M @ CMFA_MID),
    ("x3cflux (13C NA)",      x3cflux_mid_true),
]
for label, row in rows:
    print(f"{label:<35} {row[0]:9.4e} {row[1]:9.4f} {row[2]:9.4f} {row[3]:9.4e}")

print()
print("13C NA contribution (x3cflux - cmfa):")
diff = x3cflux_mid_true - CMFA_MID
for i, d in enumerate(diff):
    print(f"  M+{i}: {d:+.4e}")
print("\nExpected from 2 unlabeled C × 1.06% NA:")
print(f"  M+2 baseline: 2 × 0.0106 × (1−0.0106) ≈ {2*0.0106*(1-0.0106):.4f} ≈ 2.1%")

In [ ]:
# Plot confidence intervals
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

ci_data = [
    ("cmfa (true)",                     50.0, None,   None,   20.0, None,   None,   "#2196F3", "*"),
    ("freeflux (corrected init)",        50.0, 49.975, 50.025, 20.0, 19.901, 20.099, "#4CAF50", "o"),
    ("freeflux (bad init)",              30.0, None,   None,   30.0, None,   None,   "#FF5722", "x"),
    ("influx_si",                         50.0, 35.856, 81.572, 20.0, 17.422, 22.475, "#9C27B0", "s"),
    ("x3cflux (13C NA bias)",            32.4, None,   None,   20.0, None,   None,   "#FF9800", "^"),
]

for ax, col_start, true_val, xlabel in [
        (axes[0], 1, 50.0, "R3 (D→B)"),
        (axes[1], 4, 20.0, "R5 (B+C→D)")]:
    y = np.arange(len(ci_data))
    ax.axvline(true_val, color="k", linestyle="--", linewidth=1.5, alpha=0.6, label="Ground truth")
    for i, row in enumerate(ci_data):
        name, c, m = row[0], row[7], row[8]
        val, lo, hi = row[col_start], row[col_start+1], row[col_start+2]
        if lo is not None and hi is not None:
            ax.barh(y[i], hi-lo, left=lo, height=0.4, color=c, alpha=0.25)
            ax.plot([lo, hi], [y[i], y[i]], color=c, linewidth=2)
        ax.plot(val, y[i], color=c, marker=m, markersize=10, zorder=5,
                label=name.split("\n")[0])
    ax.set_yticks(y)
    ax.set_yticklabels([r[0] for r in ci_data], fontsize=9)
    ax.set_xlabel(f"{xlabel} (flux units)", fontsize=11)
    ax.set_title(xlabel)
    ax.grid(axis="x", alpha=0.3)

fig.suptitle("95% Confidence intervals — R3 and R5", fontsize=13)
plt.tight_layout()
plt.savefig("tool_comparison_CIs.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: tool_comparison_CIs.png")

## 7. Conclusions

1. **Network topology and atom mapping are identical across all tools.**  
   freeflux, influx_si, and x3cflux all implement D[1]=B[2], D[2]=B[3], D[3]=C[1] for R5.

2. **freeflux and influx_si recover R3=50, R5=20 exactly** when given correct starting conditions  
   and the exact cmfa true MID. The chi-squared is effectively zero.

3. **x3cflux systematically biases R3** (estimates ~32 instead of 50) because it applies  
   ¹³C natural abundance correction (~1.06 %/C) that is absent in the input data.  
   To use x3cflux correctly, input data must include ¹³C NA contamination  
   (i.e., raw experimental MS intensities, not corrected compositions).

4. **isocor corrects only H/O isotopes** (< 0.7 % M+2 effect for C₃H₆O₃).  
   The ¹³C NA effect (~2.1 % M+2) is *not* handled by isocor.  
   Workflow: raw MS → isocor → corrected MID → freeflux / influx_si.  
   For x3cflux: raw MS → x3cflux directly (internal correction).

5. **Confidence intervals differ in width:**  
   freeflux CIs are very tight (profile-likelihood, tight data); influx_si MC CIs  
   are wider (reflects under-determination: 1 MID measurement, 2 free fluxes).